# Quantum convolutional kernels vs Sobel

Small sanity check for the preprocessing front: can a 4-qubit quanvolutional kernel
pick out pipe edges as well as a plain Sobel filter? We run both on the same frames,
score them against a Canny reference, and use a Wilcoxon test to see if the gap is real
or just noise. Nothing here feeds the classifier yet, it's only to decide whether the
quantum kernel is worth carrying forward.

In [ ]:
!pip -q install pennylane gdown scipy scikit-image seaborn

In [ ]:
import os
import glob
import time
import random
import warnings
import cv2
import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pennylane as qml

from scipy.stats import wilcoxon
from sklearn.metrics import precision_recall_fscore_support, jaccard_score
from skimage.metrics import structural_similarity as ssim

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MAX_IMAGES = 50
SIZE = 64
STRIDE = 2

FOLDER_ID = "1mxkceDypm6aFGJIBp9Y3LC--qf9MZzmv"
DATA_DIR = "/content/Dataset_Final_512"


In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

if not glob.glob(os.path.join(DATA_DIR, "*")):
    gdown.download_folder(
        id=FOLDER_ID,
        output=DATA_DIR,
        quiet=False,
        use_cookies=False,
        remaining_ok=True
    )

paths = [
    p for p in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)
    if p.lower().endswith((".jpg", ".jpeg", ".png"))
]

assert paths, "No se encontraron imágenes"

rng = np.random.default_rng(SEED)
paths = list(rng.choice(paths, min(MAX_IMAGES, len(paths)), replace=False))

def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (SIZE, SIZE))
    return img.astype(np.float32) / 255

images = [load_gray(p) for p in paths]

print(f"{len(images)} imágenes cargadas")


In [ ]:
dev = qml.device("default.qubit", wires=4)

rngq = np.random.default_rng(SEED)
weights = rngq.uniform(-np.pi, np.pi, (2, 4, 2))

@qml.qnode(dev)
def qkernel(x):
    for i in range(4):
        qml.RY(np.pi*x[i], wires=i)
        qml.RX(np.pi*x[i]/2, wires=i)

    for layer in range(2):
        for i in range(4):
            qml.RX(weights[layer,i,0], wires=i)
            qml.RY(weights[layer,i,1], wires=i)

        qml.CNOT([0,1])
        qml.CNOT([1,2])
        qml.CNOT([2,3])
        qml.CNOT([3,0])

    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

def normalize(x):
    x = x - x.min()
    return x/(x.max()+1e-8)

def sobel_edges(img):
    gx = cv2.Sobel(img, cv2.CV_32F,1,0,ksize=3)
    gy = cv2.Sobel(img, cv2.CV_32F,0,1,ksize=3)
    return normalize(cv2.magnitude(gx,gy))

def canny_reference(img):
    x = (img*255).astype(np.uint8)
    return cv2.Canny(x,80,160).astype(np.float32)/255

def binary_map(img):
    return (img > img.mean()).astype(np.uint8)

def quantum_edges(img):
    out = np.zeros_like(img)

    for y in range(0,SIZE-1,STRIDE):
        for x in range(0,SIZE-1,STRIDE):
            patch = img[y:y+2,x:x+2].flatten()
            value = np.mean(qkernel(patch))
            out[y:y+2,x:x+2] = abs(value)

    return normalize(out)


In [ ]:
records = []
outputs = []

for i,(path,img) in enumerate(zip(paths,images),1):

    ref = canny_reference(img)

    t=time.perf_counter()
    sob = sobel_edges(img)
    ts=time.perf_counter()-t

    t=time.perf_counter()
    qck = quantum_edges(img)
    tq=time.perf_counter()-t

    outputs.append((img,ref,sob,qck))

    for name,edge,sec in [
        ("Sobel",sob,ts),
        ("QCK",qck,tq)
    ]:

        pred=binary_map(edge)

        _,_,f1,_ = precision_recall_fscore_support(
            ref.ravel(),
            pred.ravel(),
            average="binary",
            zero_division=0
        )

        records.append({
            "imagen":os.path.basename(path),
            "metodo":name,
            "f1":f1,
            "iou":jaccard_score(ref.ravel(),pred.ravel(),zero_division=0),
            "ssim":ssim(ref,edge,data_range=1)
        })

    print(i,"/",len(images))

df=pd.DataFrame(records)
display(df.groupby("metodo").mean(numeric_only=True))


In [ ]:
for metric in ["f1","iou","ssim"]:

    t=df.pivot(index="imagen",columns="metodo",values=metric).dropna()

    w,p=wilcoxon(t["Sobel"],t["QCK"])

    print(metric.upper())
    print("Sobel:",round(t["Sobel"].mean(),4))
    print("QCK:",round(t["QCK"].mean(),4))
    print("p:",p)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15,4))
for a, metric in zip(ax, ["f1","iou","ssim"]):
    sns.barplot(data=df, x='metodo', y=metric, errorbar='sd', ax=a, palette='viridis')
    a.set_title(metric.upper() if metric != 'segundos' else 'Tiempo medio')
plt.tight_layout(); plt.show()

In [ ]:
os.makedirs("/content/comparaciones",exist_ok=True)

for i,(img,ref,sob,qck) in enumerate(outputs,1):

    fig,ax=plt.subplots(1,4,figsize=(12,3))

    for a,x,t in zip(
        ax,
        [img,ref,sob,qck],
        ["Original","Canny","Sobel","QCK"]
    ):
        a.imshow(x,cmap="gray")
        a.set_title(t)
        a.axis("off")

    plt.tight_layout()
    plt.savefig(f"/content/comparaciones/{i}.png",dpi=300)
    plt.close()

df.to_csv("/content/resultados_QCK_Sobel.csv",index=False)

print("Finalizado")


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("comparaciones", "zip", "comparaciones")

files.download("comparaciones.zip")